# LeetCode 1930 — Unique Length-3 Palindromic Subsequences bằng Python

Link đề: https://leetcode.com/problems/unique-length-3-palindromic-subsequences/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def countPalindromicSubsequence(self, s: str) -> int:
        ...
```

Lộ trình: **mọi bộ ba chỉ số `i < j < k`** `O(n³)` → **cố định chữ cái ngoài, lấy lần xuất hiện đầu & cuối, đếm chữ khác nhau ở giữa** `O(26·n)` (lời giải nộp) → **tổng tiền tố số lần xuất hiện** `O(26·n + 26²)` → **bitmask chữ cái bên trái + đếm bên phải tại mỗi vị trí giữa**.

## 1. Mục tiêu học

1. Nhận ra palindrome độ dài 3 có dạng `x y x` — chỉ phụ thuộc **cặp (chữ ngoài `x`, chữ giữa `y`)**, tối đa `26 × 26` khả năng.
2. Dùng lập luận **"mở rộng tối đa"**: với chữ ngoài `x`, chọn lần xuất hiện **đầu tiên** và **cuối cùng** cho khoảng giữa rộng nhất.
3. Đếm số chữ **khác nhau** trong một đoạn bằng `set` hoặc **tổng tiền tố theo từng chữ cái**.
4. Phân biệt "đếm **cách** chọn" với "đếm **chuỗi khác nhau**".
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, trả về số **palindrome độ dài 3 khác nhau** là **dãy con** của `s`. Cùng một chuỗi nhận được bằng nhiều cách vẫn chỉ đếm **một lần**.

**Ví dụ 1:** `s = "aabca"` → `3` (`"aba"`, `"aaa"`, `"aca"`).

**Ví dụ 2:** `s = "adc"` → `0`.

**Ví dụ 3:** `s = "bbcbaba"` → `4` (`"bbb"`, `"bcb"`, `"bab"`, `"aba"`).

**Ràng buộc:** `3 ≤ s.length ≤ 10⁵`, chỉ chữ thường.

In [1]:
import random
import string
import time
from itertools import combinations
from typing import List

examples = [("aabca", 3), ("adc", 0), ("bbcbaba", 4)]
for s, want in examples:
    print(repr(s), "-> kỳ vọng", want)

'aabca' -> kỳ vọng 3
'adc' -> kỳ vọng 0
'bbcbaba' -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1 Dãy con và palindrome độ dài 3

Dãy con độ dài 3 = chọn ba chỉ số `i < j < k`. Là palindrome ⇔ `s[i] == s[k]` (ký tự giữa tùy ý).

In [2]:
s = "aabca"
for i, j, k in combinations(range(len(s)), 3):
    t = s[i] + s[j] + s[k]
    if t == t[::-1]:
        print((i, j, k), t)

(0, 1, 4) aaa
(0, 2, 4) aba
(0, 3, 4) aca
(1, 2, 4) aba
(1, 3, 4) aca


### 3.2 `str.find` và `str.rfind`

`s.find(c)` = chỉ số xuất hiện **đầu tiên**, `s.rfind(c)` = **cuối cùng** (hoặc `-1`).

In [3]:
s = "bbcbaba"
for c in "abc":
    print(c, "đầu:", s.find(c), "cuối:", s.rfind(c))

a đầu: 4 cuối: 6
b đầu: 0 cuối: 5
c đầu: 2 cuối: 2


### 3.3 Số chữ khác nhau trong một đoạn: `len(set(...))`

In [4]:
s = "bbcbaba"
print(set(s[1:6]), len(set(s[1:6])))

{'b', 'a', 'c'} 3


### 3.4 Tổng tiền tố theo từng chữ cái

`pre[i][c]` = số lần chữ `c` xuất hiện trong `s[:i]`. Đoạn `s[l:r]` chứa `c` ⇔ `pre[r][c] − pre[l][c] > 0` — tra `O(1)`.

In [5]:
s = "aabca"
pre = [[0] * 26]
for ch in s:
    row = pre[-1][:]
    row[ord(ch) - 97] += 1
    pre.append(row)
l, r = 1, 4
print("đoạn", repr(s[l:r]), "có 'b'?", pre[r][1] - pre[l][1] > 0, "| có 'a'?", pre[r][0] - pre[l][0] > 0)

đoạn 'abc' có 'b'? True | có 'a'? True


### 3.5 Bitmask tập chữ cái

Tập chữ cái biểu diễn bằng số 26 bit: thêm chữ `c` bằng `mask |= 1 << c`; kiểm `mask >> c & 1`.

In [6]:
mask = 0
for ch in "bca":
    mask |= 1 << (ord(ch) - 97)
print(bin(mask), [chr(97 + c) for c in range(26) if mask >> c & 1])

0b111 ['a', 'b', 'c']


## 4. Ý tưởng nền tảng / trực giác

Mỗi palindrome độ dài 3 là `x y x`. Cố định chữ ngoài `x`:
- Cần một `x` ở bên trái và một `x` ở bên phải, với `y` nằm **giữa**.
- Muốn **nhiều** lựa chọn `y` nhất ⇒ lấy `x` **đầu tiên** (`L`) và `x` **cuối cùng** (`R`) — khoảng giữa `s[L+1 : R]` rộng nhất, chứa mọi khả năng.
- Số palindrome dạng `x ? x` khác nhau = số chữ **khác nhau** trong `s[L+1 : R]`.

Ví dụ 3 `s = "bbcbaba"`:

| `x` | đầu `L` | cuối `R` | đoạn giữa | chữ khác nhau |
|---|---|---|---|---|
| `a` | 4 | 6 | `"b"` | `{b}` ⇒ `aba` |
| `b` | 0 | 5 | `"bcba"` | `{a, b, c}` ⇒ `bab, bbb, bcb` |
| `c` | 2 | 2 | — | ∅ |

Tổng `1 + 3 = 4`. ✔

In [7]:
s = "bbcbaba"
for c in sorted(set(s)):
    L, R = s.find(c), s.rfind(c)
    mid = set(s[L + 1:R])
    print(c, L, R, repr(s[L + 1:R]), "->", sorted(c + y + c for y in mid))

a 4 6 'b' -> ['aba']
b 0 5 'bcba' -> ['bab', 'bbb', 'bcb']
c 2 2 '' -> []


## 5. Cách 1 — Brute Force / baseline: mọi bộ ba `i < j < k`

In [8]:
def count_brute(s):
    found = set()
    n = len(s)
    for i in range(n):
        for k in range(i + 2, n):
            if s[i] == s[k]:
                for j in range(i + 1, k):
                    found.add(s[i] + s[j] + s[k])
    return len(found)

for s, want in examples:
    print(repr(s), count_brute(s), "| kỳ vọng", want)

'aabca' 3 | kỳ vọng 3
'adc' 0 | kỳ vọng 0
'bbcbaba' 4 | kỳ vọng 4


**Độ phức tạp:** `O(n³)` — `n = 10⁵` là bất khả. **Nút thắt:** cùng một palindrome `x y x` bị sinh ra nhiều lần; với mỗi `x` chỉ cần **một** cặp đầu–cuối rộng nhất.

In [9]:
w = "".join(random.Random(0).choice("abc") for _ in range(150))
t = time.perf_counter(); count_brute(w); dt = time.perf_counter() - t
print(f"n=150: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * (10**5 / 150) ** 3 / 3600 / 24 / 365:.0f} năm")

n=150: 0.05s -> n=10^5 ước tính ≈ 1 năm


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** palindrome `x y x` là dãy con của `s` ⇔ tồn tại `i < j < k` với `s[i] = s[k] = x`, `s[j] = y` ⇔ `y` xuất hiện trong `s[first(x) + 1 : last(x)]`.

*Chiều ⇐ hiển nhiên. Chiều ⇒:* nếu `i < j < k` tồn tại thì `first(x) ≤ i < j < k ≤ last(x)` nên `j` nằm trong đoạn giữa.

**Quy tắc:** đáp án `= Σ_{x} |{chữ cái trong s[first(x)+1 : last(x)]}|`, chỉ xét `x` có `first(x) < last(x)`.

**Tối ưu:** đếm chữ khác nhau trong đoạn bằng `set` (`O(độ dài)`) hoặc tổng tiền tố (`O(26)`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đầu & cuối của mỗi chữ + `set` đoạn giữa (`O(26 · n)`) — lời giải nộp

In [10]:
class SolutionFirstLast:
    def countPalindromicSubsequence(self, s: str) -> int:
        ans = 0
        for c in set(s):
            L, R = s.find(c), s.rfind(c)
            if R - L > 1:
                ans += len(set(s[L + 1:R]))
        return ans

for s, want in examples:
    print(SolutionFirstLast().countPalindromicSubsequence(s), want)

3 3
0 0
4 4


**Độ phức tạp:** tối đa 26 chữ ngoài, mỗi chữ dựng `set` của đoạn `O(n)` ⇒ `O(26·n)`; bộ nhớ `O(n)` cho lát cắt (có thể tránh bằng Cách 3).

### 7.2 Cách 3 — Tổng tiền tố theo chữ cái (`O(26·n + 26²)`)

In [11]:
class SolutionPrefix:
    def countPalindromicSubsequence(self, s: str) -> int:
        n = len(s)
        first = [-1] * 26; last = [-1] * 26
        pre = [[0] * (n + 1) for _ in range(26)]          # pre[c][i] = số lần c trong s[:i]
        for i, ch in enumerate(s):
            c = ord(ch) - 97
            if first[c] == -1: first[c] = i
            last[c] = i
            for d in range(26):
                pre[d][i + 1] = pre[d][i]
            pre[c][i + 1] += 1
        ans = 0
        for x in range(26):
            L, R = first[x], last[x]
            if L != -1 and R - L > 1:
                ans += sum(1 for y in range(26) if pre[y][R] - pre[y][L + 1] > 0)
        return ans

for s, want in examples:
    print(SolutionPrefix().countPalindromicSubsequence(s), want)

3 3
0 0
4 4


**Độ phức tạp:** dựng tiền tố `O(26·n)`, trả lời `26 × 26` truy vấn `O(1)`. Bộ nhớ `O(26·n)` — lớn hơn Cách 2 nhưng mở rộng tốt khi cần **nhiều** truy vấn đoạn.

### 7.3 Cách 4 — Duyệt vị trí giữa: bitmask trái + đếm phải

Với mỗi vị trí giữa `j`, tập chữ đã xuất hiện **bên trái** (bitmask `left`) và **bên phải** (mảng đếm `right`). Mọi `x` có mặt ở **cả hai** phía cho palindrome `x s[j] x`. Gom các cặp `(x, s[j])` vào một tập (hoặc mảng bitmask theo chữ giữa).

In [12]:
class SolutionMiddle:
    def countPalindromicSubsequence(self, s: str) -> int:
        right = [0] * 26
        for ch in s:
            right[ord(ch) - 97] += 1
        right_mask = 0
        for c in range(26):
            if right[c]: right_mask |= 1 << c
        left_mask = 0
        seen = [0] * 26                                   # seen[y] = bitmask các x đã ghép với chữ giữa y
        for ch in s:
            y = ord(ch) - 97
            right[y] -= 1
            if right[y] == 0:
                right_mask &= ~(1 << y)
            seen[y] |= left_mask & right_mask             # x có mặt cả trái lẫn phải
            left_mask |= 1 << y
        return sum(bin(m).count("1") for m in seen)

for s, want in examples:
    print(SolutionMiddle().countPalindromicSubsequence(s), want)

3 3
0 0
4 4


**Độ phức tạp:** `O(n)` thao tác bit (mỗi vị trí `O(1)` với số 26 bit) + `O(26)` cuối ⇒ `O(n)`; bộ nhớ `O(26)`. Đây là cách **một lượt**, không cần lát cắt.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 `"aabca"` (chỉ ghi các chữ `a, b, c`):

| j | `s[j]` | `left` trước | `right` sau khi bỏ `s[j]` | `left ∩ right` | ghép |
|---|---|---|---|---|---|
| 0 | a | ∅ | {a, b, c} | ∅ | — |
| 1 | a | {a} | {a, b, c} | {a} | `aaa` |
| 2 | b | {a} | {a, c} | {a} | `aba` |
| 3 | c | {a, b} | {a} | {a} | `aca` |
| 4 | a | {a, b, c} | ∅ | ∅ | — |

Tổng 3. ✔

In [13]:
def trace_middle(s):
    right = {}
    for ch in s: right[ch] = right.get(ch, 0) + 1
    left = set(); found = set()
    for j, ch in enumerate(s):
        right[ch] -= 1
        rset = {c for c, v in right.items() if v > 0}
        both = left & rset
        new = {x + ch + x for x in both} - found
        found |= new
        print(f"j={j} '{ch}': trái={sorted(left)} phải={sorted(rset)} chung={sorted(both)} -> mới {sorted(new)}")
        left.add(ch)
    print("tổng:", len(found), sorted(found))
    return len(found)

trace_middle("aabca")

j=0 'a': trái=[] phải=['a', 'b', 'c'] chung=[] -> mới []
j=1 'a': trái=['a'] phải=['a', 'b', 'c'] chung=['a'] -> mới ['aaa']
j=2 'b': trái=['a'] phải=['a', 'c'] chung=['a'] -> mới ['aba']
j=3 'c': trái=['a', 'b'] phải=['a'] chung=['a'] -> mới ['aca']
j=4 'a': trái=['a', 'b', 'c'] phải=[] chung=[] -> mới []
tổng: 3 ['aaa', 'aba', 'aca']


3

In [14]:
trace_middle("bbcbaba")

j=0 'b': trái=[] phải=['a', 'b', 'c'] chung=[] -> mới []
j=1 'b': trái=['b'] phải=['a', 'b', 'c'] chung=['b'] -> mới ['bbb']
j=2 'c': trái=['b'] phải=['a', 'b'] chung=['b'] -> mới ['bcb']
j=3 'b': trái=['b', 'c'] phải=['a', 'b'] chung=['b'] -> mới []
j=4 'a': trái=['b', 'c'] phải=['a', 'b'] chung=['b'] -> mới ['bab']
j=5 'b': trái=['a', 'b', 'c'] phải=['a'] chung=['a'] -> mới ['aba']
j=6 'a': trái=['a', 'b', 'c'] phải=[] chung=[] -> mới []
tổng: 4 ['aba', 'bab', 'bbb', 'bcb']


4

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def countPalindromicSubsequence(self, s: str) -> int:
        ans = 0
        for c in set(s):                       # chữ ở hai đầu
            first, last = s.find(c), s.rfind(c)
            if last - first > 1:
                ans += len(set(s[first + 1:last]))   # mỗi chữ giữa khác nhau cho một palindrome c?c
        return ans

for s, want in examples:
    assert Solution().countPalindromicSubsequence(s) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `P` là tập palindrome độ dài 3 khác nhau là dãy con của `s`. Mỗi phần tử của `P` có dạng `x y x` và được xác định **duy nhất** bởi cặp `(x, y)`.

**Bổ đề:** `x y x ∈ P` ⇔ `first(x) < last(x)` và `y` xuất hiện tại một vị trí `j` với `first(x) < j < last(x)`.
- *(⇐)* chọn `i = first(x)`, `j`, `k = last(x)`.
- *(⇒)* nếu `s[i] s[j] s[k] = x y x` với `i < j < k` thì `first(x) ≤ i < j < k ≤ last(x)`.

Vậy `|P| = Σ_x |{y : y ∈ s[first(x)+1 : last(x)]}|` — đúng thứ lời giải tính. Các cặp `(x, y)` khác nhau cho chuỗi khác nhau ⇒ không đếm trùng.

**Cách 4:** tại vị trí giữa `j`, `x ∈ left ∩ right` ⇔ có `i < j < k` với `s[i] = s[k] = x` ⇒ ghép đúng cặp `(x, s[j])`; hợp qua mọi `j` cho đúng `P` (bitmask `seen[y]` khử trùng).

Kiểm chứng Bổ đề bằng máy: với mọi cặp chữ `(x, y)`, "tồn tại `i < j < k` với `s[i] = s[k] = x`, `s[j] = y`" ⇔ "`y` có trong `s[first(x)+1 : last(x)]`".

In [16]:
rng_l = random.Random(13)
for _ in range(300):
    s_ = "".join(rng_l.choice("abcd") for _ in range(rng_l.randint(3, 15)))
    for x in "abcd":
        L, R = s_.find(x), s_.rfind(x)
        for y in "abcd":
            exists = any(s_[i] == x and s_[j] == y and s_[k] == x for i, j, k in combinations(range(len(s_)), 3))
            by_lemma = L != -1 and R - L > 1 and y in s_[L + 1:R]
            assert exists == by_lemma
print("Bổ đề đúng trên 300 chuỗi × 16 cặp chữ")

Bổ đề đúng trên 300 chuỗi × 16 cặp chữ


## 11. Phân tích Time / Space

Gọi `n = len(s) ≤ 10⁵`, `σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi bộ ba | `O(n³)` | `O(σ²)` |
| 2. Đầu/cuối + set | `O(σ · n)` | `O(n)` |
| 3. Tiền tố | `O(σ · n + σ²)` | `O(σ · n)` |
| 4. Giữa + bitmask | `O(n + σ)` | `O(σ)` |

In [17]:
rng = random.Random(1930)
big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))
res = []
for name, sol in [("Đầu/cuối/Nộp", Solution()), ("Tiền tố", SolutionPrefix()), ("Giữa+bitmask", SolutionMiddle())]:
    t = time.perf_counter(); r = sol.countPalindromicSubsequence(big); res.append(r)
    print(f"{name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
assert len(set(res)) == 1

Đầu/cuối/Nộp: 676 trong 17 ms


Tiền tố: 676 trong 112 ms


Giữa+bitmask: 676 trong 14 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [18]:
sols = [Solution(), SolutionFirstLast(), SolutionPrefix(), SolutionMiddle()]
edge = examples + [
    ("aaa", 1),
    ("aba", 1),
    ("abc", 0),
    ("abcba", 3),            # aba, aca, bcb
    ("aaaa", 1),
    ("abab", 2),             # aba, bab
    ("abcdefghijklmnopqrstuvwxyza", 25),
]
for s, want in edge:
    assert count_brute(s) == want, (s, count_brute(s))
    for sol in sols:
        assert sol.countPalindromicSubsequence(s) == want, (s, sol)
    print("OK", repr(s), "->", want)

OK 'aabca' -> 3
OK 'adc' -> 0
OK 'bbcbaba' -> 4
OK 'aaa' -> 1
OK 'aba' -> 1
OK 'abc' -> 0
OK 'abcba' -> 3
OK 'aaaa' -> 1
OK 'abab' -> 2
OK 'abcdefghijklmnopqrstuvwxyza' -> 25


### 12.2 Ngẫu nhiên có seed: so với vét cạn

In [19]:
rng = random.Random(42)
for _ in range(800):
    alpha = rng.choice(["ab", "abc", "abcde", string.ascii_lowercase])
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(3, 40)))
    want = count_brute(s)
    for sol in sols:
        assert sol.countPalindromicSubsequence(s) == want, s
print("800 chuỗi ngẫu nhiên: 4 cách khớp vét cạn")

800 chuỗi ngẫu nhiên: 4 cách khớp vét cạn


### 12.3 Cận trên: tối đa `26 × 26 = 676`

(Lặp `a..z` **hai** lần chỉ được `26 × 25 = 650` — chữ ngoài `x` không có bản sao nào nằm giữa hai lần xuất hiện của chính nó; cần lặp ba lần.)

In [20]:
full = string.ascii_lowercase * 3
print("chuỗi 'a..z' lặp 3 lần:", Solution().countPalindromicSubsequence(full))
assert Solution().countPalindromicSubsequence(full) == 26 * 26

chuỗi 'a..z' lặp 3 lần: 676


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm số **cách** chọn thay vì số chuỗi khác nhau

In [21]:
def wrong_count_ways(s):
    return sum(1 for i, j, k in combinations(range(len(s)), 3) if s[i] == s[k])   # SAI

for s, want in examples:
    print(repr(s), "sai:", wrong_count_ways(s), "| đúng:", want)

'aabca' sai: 5 | đúng: 3
'adc' sai: 0 | đúng: 0
'bbcbaba' sai: 12 | đúng: 4


**Vì sao sai:** `"aabca"` có nhiều cách chọn ra `"aba"` (`a` ở chỉ số 0 hoặc 1). Đề yêu cầu **chuỗi khác nhau**. **Sửa:** đếm cặp `(x, y)` khác nhau.

### Lỗi 2 — Đoạn giữa bao gồm cả hai đầu

In [22]:
def wrong_inclusive(s):
    ans = 0
    for c in set(s):
        L, R = s.find(c), s.rfind(c)
        if R > L:
            ans += len(set(s[L:R + 1]))                 # SAI: gồm cả s[L], s[R]
    return ans

for s in ["aab", "abca", "aabca"]:
    print(repr(s), "sai:", wrong_inclusive(s), "| đúng:", Solution().countPalindromicSubsequence(s))

'aab' sai: 1 | đúng: 0
'abca' sai: 3 | đúng: 2
'aabca' sai: 3 | đúng: 3


**Vì sao sai:** chữ giữa phải nằm **giữa** hai chữ ngoài. Với `"aab"`: `first(a) = 0`, `last(a) = 1`, không có vị trí ở giữa ⇒ không có `"aaa"`; bản sai đếm `set("aa") = {a}` ⇒ 1. **Sửa:** `s[L+1 : R]` và điều kiện `R − L > 1`.

### Lỗi 3 — Chỉ xét **các cặp kề nhau** của chữ ngoài

In [23]:
def wrong_adjacent_pairs(s):
    found = set()
    pos = {}
    for i, ch in enumerate(s):
        pos.setdefault(ch, []).append(i)
    for c, ps in pos.items():
        for a, b in zip(ps, ps[1:]):                    # SAI: chỉ giữa hai lần xuất hiện liên tiếp
            for y in set(s[a + 1:b]):
                found.add(c + y + c)
    return len(found)

for s in ["abacba", "bbcbaba", "abcab"]:
    print(repr(s), "sai:", wrong_adjacent_pairs(s), "| đúng:", Solution().countPalindromicSubsequence(s))

'abacba' sai: 4 | đúng: 5
'bbcbaba' sai: 3 | đúng: 4
'abcab' sai: 4 | đúng: 4


**Vì sao sai:** đoạn giữa hai lần xuất hiện **liên tiếp** của `x` **không bao giờ chứa `x`** — nên mọi palindrome `x x x` bị bỏ sót. `"abacba"` thiếu `"aaa"` (ra 4 thay vì 5), `"bbcbaba"` thiếu `"bbb"` (ra 3 thay vì 4). Với `"abcab"` (không có chữ nào xuất hiện 3 lần) thì tình cờ đúng. **Sửa:** dùng lần xuất hiện **đầu tiên và cuối cùng** — đoạn giữa của chúng là **hợp** của mọi đoạn giữa các cặp kề, cộng thêm chính các lần xuất hiện ở giữa của `x`.

### Lỗi 4 — `O(n²)` theo từng cặp vị trí ngoài

In [24]:
def slow_pairs(s):
    found = set()
    n = len(s)
    for i in range(n):
        seen_mid = set()
        for k in range(i + 1, n):
            if s[k] == s[i]:
                for y in seen_mid:
                    found.add(s[i] + y + s[i])
            seen_mid.add(s[k])
    return len(found)

w = "".join(random.Random(1).choice(string.ascii_lowercase) for _ in range(1500))
t = time.perf_counter(); r = slow_pairs(w); dt = time.perf_counter() - t
assert r == Solution().countPalindromicSubsequence(w)
print(f"n=1500: {dt:.2f}s (đúng) -> n=10^5 ≈ {dt * (10**5 / 1500) ** 2:.0f}s")

n=1500: 0.14s (đúng) -> n=10^5 ≈ 620s


**Vì sao chưa tốt:** đúng nhưng `O(n² · σ)`. Lập luận "đầu–cuối" bỏ được vòng lặp theo điểm đầu `i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Palindrome độ dài cố định nhỏ** ⇒ liệt kê theo **cấu trúc** (`x y x`, `x y z y x`), không theo vị trí.
- **"Tồn tại `i < j < k`"** ⇒ dùng **cực trị**: vị trí đầu tiên/cuối cùng cho khoảng rộng nhất.
- **Đếm chữ khác nhau trong đoạn** ⇒ `set`, bitmask, hoặc tiền tố theo chữ cái.
- "Đếm chuỗi **khác nhau**" ⇒ đếm theo **khóa định danh** (ở đây cặp `(x, y)`), không theo cách chọn.

## 15. Biến thể của bài

1. **Đếm số cách** (không khử trùng) chọn palindrome độ dài 3 ⇒ với mỗi vị trí giữa `j`: `Σ_x left[x]·right[x]`.
2. **LeetCode 2484 — Count Palindromic Subsequences** (độ dài 5, đếm số cách, mod) ⇒ DP với tiền tố/hậu tố cặp chữ.
3. **Palindrome độ dài 5 khác nhau** (Bài tập 5) ⇒ cặp ngoài `(x, y)` + chữ giữa `z`, dùng "đầu–cuối tham lam".
4. **Truy vấn trên đoạn `[l, r]`** ⇒ tiền tố theo chữ cái + vị trí đầu/cuối trong đoạn (bisect).
5. **LeetCode 516 / 730** — dãy con palindrome dài nhất / đếm mọi dãy con palindrome khác nhau (DP đoạn).

Minh họa biến thể 4 — **nhiều truy vấn trên đoạn** `[l, r]`: với mỗi truy vấn, cần `first(x)`, `last(x)` **trong đoạn** — tìm bằng `bisect` trên danh sách vị trí của `x`; rồi đếm chữ có mặt ở khoảng giữa bằng tiền tố theo chữ cái. Mỗi truy vấn `O(26 · (log n + 26))`. (Ví dụ tự dựng, đối chiếu bằng lời giải gốc trên chuỗi con.)

In [25]:
from bisect import bisect_left, bisect_right

class RangePalindromes:
    def __init__(self, s):
        self.s = s
        self.pos = [[] for _ in range(26)]
        for i, ch in enumerate(s):
            self.pos[ord(ch) - 97].append(i)
        self.pre = [[0] * (len(s) + 1) for _ in range(26)]
        for c in range(26):
            row = self.pre[c]
            for i, ch in enumerate(s):
                row[i + 1] = row[i] + (ord(ch) - 97 == c)

    def query(self, l, r):                      # đoạn s[l..r] (đóng)
        ans = 0
        for x in range(26):
            p = self.pos[x]
            a = bisect_left(p, l); b = bisect_right(p, r) - 1
            if a < len(p) and b >= 0 and a < b and p[b] - p[a] > 1:
                L, R = p[a], p[b]
                ans += sum(1 for y in range(26) if self.pre[y][R] - self.pre[y][L + 1] > 0)
        return ans

rng_q = random.Random(21)
w = "".join(rng_q.choice("abcde") for _ in range(300))
rp = RangePalindromes(w)
for _ in range(300):
    l = rng_q.randrange(len(w)); r = rng_q.randrange(l, len(w))
    assert rp.query(l, r) == Solution().countPalindromicSubsequence(w[l:r + 1])
print("300 truy vấn đoạn: khớp lời giải gốc trên chuỗi con; ví dụ query(0, 9) =", rp.query(0, 9))

300 truy vấn đoạn: khớp lời giải gốc trên chuỗi con; ví dụ query(0, 9) = 8


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi bộ ba | vét cạn | `O(n³)` | `O(σ²)` | kiểm chứng |
| 2. Đầu/cuối + set | cực trị + set đoạn giữa | `O(σn)` | `O(n)` | **nộp bài**, ngắn |
| 3. Tiền tố | đếm có mặt trong đoạn `O(1)` | `O(σn + σ²)` | `O(σn)` | nhiều truy vấn đoạn |
| 4. Giữa + bitmask | một lượt, trái ∩ phải | `O(n)` | `O(σ)` | tối ưu bộ nhớ |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `"aaa"`, `"abcba"`, `"abab"`?

**Bài 2 (tính tay):** Lập bảng đầu/cuối và đoạn giữa cho `"abacaba"`.

**Bài 3 (điền TODO):** Viết `list_palindromes(s)` trả về **danh sách đã sắp xếp** các palindrome độ dài 3 khác nhau.

**Bài 4 (tự viết lại):** Viết `count_ways_len3(s)` — số **cách** chọn `(i, j, k)` tạo palindrome độ dài 3 (không khử trùng), `O(26·n)`.

**Bài 5 (mini challenge):** Viết `count_unique_len5(s)` — số palindrome độ dài 5 **khác nhau** là dãy con của `s`.

In [26]:
def list_palindromes(s):
    # TODO
    pass

def count_ways_len3(s):
    # TODO
    pass

def count_unique_len5(s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Áp bảng đầu/cuối | `"aaa"`: 1 | `"abcba"`: 3, `"abab"`: 2 |
| 2 | `a`: 0..6, `b`: 1..5, `c`: 3..3 | Giữa `a`: `{a, b, c}` | Giữa `b`: `{a, c}` ⇒ tổng 5 |
| 3 | Như lời giải nộp nhưng gom chuỗi | `c + y + c` | `sorted(...)` |
| 4 | Duyệt vị trí giữa `j` | Giữ `left[x]`, `right[x]` | Cộng `Σ_x left[x]·right[x]` |
| 5 | Dạng `x y z y x` | Với cặp `(x, y)`: lấy `x` đầu tiên, `y` đầu tiên sau nó; `x` cuối cùng, `y` cuối cùng trước nó | Đếm `z` khác nhau trong khoảng giữa hai `y` (tiền tố) |

## 19. Đáp án tham khảo

In [27]:
r1 = [Solution().countPalindromicSubsequence(s) for s in ["aaa", "abcba", "abab"]]
print("Bài 1:", r1); assert r1 == [1, 3, 2]
s2 = "abacaba"
for c in sorted(set(s2)):
    L, R = s2.find(c), s2.rfind(c)
    print("Bài 2:", c, L, R, sorted(set(s2[L + 1:R])))
assert Solution().countPalindromicSubsequence(s2) == 5 == count_brute(s2)

Bài 1: [1, 3, 2]
Bài 2: a 0 6 ['a', 'b', 'c']
Bài 2: b 1 5 ['a', 'c']
Bài 2: c 3 3 []


In [28]:
def list_palindromes(s):
    out = []
    for c in set(s):
        L, R = s.find(c), s.rfind(c)
        if R - L > 1:
            out += [c + y + c for y in set(s[L + 1:R])]
    return sorted(out)

def count_ways_len3(s):
    right = [0] * 26
    for ch in s: right[ord(ch) - 97] += 1
    left = [0] * 26; total = 0
    for ch in s:
        y = ord(ch) - 97
        right[y] -= 1
        total += sum(left[x] * right[x] for x in range(26))
        left[y] += 1
    return total

rng = random.Random(3)
for _ in range(400):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(3, 25)))
    brute_set = sorted({s[i] + s[j] + s[k] for i, j, k in combinations(range(len(s)), 3) if s[i] == s[k]})
    assert list_palindromes(s) == brute_set
    assert count_ways_len3(s) == wrong_count_ways(s)
print("Bài 3, 4: OK", list_palindromes("bbcbaba"), count_ways_len3("aabca"))

Bài 3, 4: OK ['aba', 'bab', 'bbb', 'bcb'] 5


In [29]:
def count_unique_len5(s):
    n = len(s)
    nxt = [[n] * 26 for _ in range(n + 1)]           # nxt[i][c] = vị trí nhỏ nhất >= i có chữ c
    for i in range(n - 1, -1, -1):
        nxt[i] = nxt[i + 1][:]
        nxt[i][ord(s[i]) - 97] = i
    prv = [[-1] * 26 for _ in range(n + 1)]          # prv[i][c] = vị trí lớn nhất < i có chữ c
    for i in range(n):
        prv[i + 1] = prv[i][:]
        prv[i + 1][ord(s[i]) - 97] = i
    pre = [[0] * 26]
    for ch in s:
        row = pre[-1][:]; row[ord(ch) - 97] += 1; pre.append(row)
    ans = 0
    for x in range(26):
        i = nxt[0][x]; m = prv[n][x]
        if i >= n or m <= i:
            continue
        for y in range(26):
            j = nxt[i + 1][y]; l = prv[m][y]
            if j < l - 1:                              # cần chỗ cho z giữa j và l
                ans += sum(1 for z in range(26) if pre[l][z] - pre[j + 1][z] > 0)
    return ans

def unique_len5_brute(s):
    out = set()
    for idx in combinations(range(len(s)), 5):
        t = "".join(s[i] for i in idx)
        if t == t[::-1]:
            out.add(t)
    return len(out)

rng = random.Random(9)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(5, 14)))
    assert count_unique_len5(s) == unique_len5_brute(s), s
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", count_unique_len5("abcbaabcba"))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 9


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = 0
for c in set(s):
    L, R = s.find(c), s.rfind(c)
    if R - L > 1:
        ans += len(set(s[L + 1:R]))
return ans
```

**Checklist:**
- [ ] Đếm chuỗi **khác nhau** (cặp `(x, y)`), không đếm cách chọn?
- [ ] Đoạn giữa **không gồm** hai đầu: `s[L+1:R]`?
- [ ] Dùng **lần đầu và lần cuối** của chữ ngoài (không phải cặp kề nhau)?
- [ ] Chữ giữa có thể **trùng** chữ ngoài (`"aaa"` hợp lệ)?
- [ ] `O(26·n)`.